# Register API from Text Documentation

This notebook demonstrates how to use the `register_from_text` workflow to convert raw API documentation into a registered government API contract.

## Workflow Steps
1. Parse API documentation text
2. Convert to structured contract using LLM
3. Validate the contract
4. Register with the API registry

## Setup
Import the necessary modules and configure logging to see the workflow progress.

In [1]:
import sys
import json
import logging
from pathlib import Path

# Add the parent directory to the path to import app modules
sys.path.insert(0, str(Path.cwd().parent))

from app.workflows.register_from_text import register_from_text

# Configure logging to see workflow progress
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("✓ Setup complete")

✓ Setup complete


## Example 1: Register a Simple API

Let's start with a simple example of API documentation.

In [3]:
# Example API documentation text
sample_api_docs = """
Weather API Documentation

Base URL: https://api.weather.gov

Endpoint: /v1/forecast
Method: GET
Description: Get weather forecast for a specific location

Parameters:
- lat (required): Latitude coordinate
- lon (required): Longitude coordinate
- units (optional): Unit system (metric or imperial), defaults to metric

Response Format: JSON
Content-Type: application/json

Example Request:
GET https://api.weather.gov/v1/forecast?lat=40.7128&lon=-74.0060&units=imperial

Example Response:
{
  "temperature": 72,
  "conditions": "sunny",
  "humidity": 45
}
"""

print("API Documentation:")
print(sample_api_docs)

API Documentation:

Weather API Documentation

Base URL: https://api.weather.gov

Endpoint: /v1/forecast
Method: GET
Description: Get weather forecast for a specific location

Parameters:
- lat (required): Latitude coordinate
- lon (required): Longitude coordinate
- units (optional): Unit system (metric or imperial), defaults to metric

Response Format: JSON
Content-Type: application/json

Example Request:
GET https://api.weather.gov/v1/forecast?lat=40.7128&lon=-74.0060&units=imperial

Example Response:
{
  "temperature": 72,
  "conditions": "sunny",
  "humidity": 45
}



In [4]:
# Call the workflow to register the API
result = register_from_text(sample_api_docs)

# Parse and display the result
result_data = json.loads(result)
print("\nRegistration Result:")
print(json.dumps(result_data, indent=2))

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://localhost:8080/api/v1/gov/apis "HTTP/1.1 201 Created"



Registration Result:
{
  "name": "Get Weather Forecast",
  "baseUrl": "https://api.weather.gov/",
  "httpMethod": "GET",
  "headers": [
    {
      "key": "Accept",
      "value": "application/json"
    }
  ],
  "queryParams": [
    {
      "key": "lat",
      "type": "FLOAT",
      "description": "Latitude coordinate",
      "exampleValue": "40.7128"
    },
    {
      "key": "lon",
      "type": "FLOAT",
      "description": "Longitude coordinate",
      "exampleValue": "-74.0060"
    },
    {
      "key": "units",
      "type": "STRING",
      "description": "Unit system (metric or imperial), defaults to metric",
      "exampleValue": "imperial"
    }
  ],
  "description": "Get weather forecast for a specific location"
}


## Example 2: Check for Validation Errors

Let's see what happens when we provide invalid or incomplete documentation.

In [5]:
# Empty documentation - should fail validation
empty_docs = ""

result = register_from_text(empty_docs)
result_data = json.loads(result)

print("Result with empty documentation:")
print(json.dumps(result_data, indent=2))

if "validation_errors" in result_data:
    print("\n❌ Validation failed as expected")
    for error in result_data["validation_errors"]:
        print(f"  - {error}")

Result with empty documentation:
{
  "validation_errors": [
    "source_text is required to register an API."
  ]
}

❌ Validation failed as expected
  - source_text is required to register an API.


## Example 3: Register Your Own API

Now you can try registering your own API by pasting documentation below.

In [ ]:
# Replace this with your own API documentation
your_api_docs = """
Paste your API documentation here...

Include:
- Base URL
- Endpoint path
- HTTP method
- Description
- Parameters
- Response format
"""

# Uncomment the lines below when you're ready to test
# result = register_from_text(your_api_docs)
# result_data = json.loads(result)
# print(json.dumps(result_data, indent=2))

## Helper Function: Check Registration Status

This helper function checks if the registration was successful or if there were errors.

In [ ]:
def check_registration_status(result_json: str) -> None:
    """Display registration status in a user-friendly format."""
    result_data = json.loads(result_json)
    
    if "validation_errors" in result_data:
        print("❌ Registration Failed")
        print("\nErrors:")
        for error in result_data["validation_errors"]:
            print(f"  • {error}")
        
        if "llm_response" in result_data:
            print("\nLLM Response (for debugging):")
            print(result_data["llm_response"])
    else:
        print("✅ Registration Successful!")
        print("\nRegistered API Contract:")
        print(json.dumps(result_data, indent=2))
        
        # Display key information
        if "name" in result_data:
            print(f"\nAPI Name: {result_data['name']}")
        if "base_url" in result_data:
            print(f"Base URL: {result_data['base_url']}")
        if "endpoints" in result_data:
            print(f"Endpoints: {len(result_data['endpoints'])}")

# Example usage
# check_registration_status(result)

## Summary

The `register_from_text` workflow provides a simple way to convert API documentation into registered contracts.

### Key Features:
- ✅ Automated parsing of API documentation
- ✅ LLM-powered structure extraction
- ✅ Schema validation
- ✅ Automatic registry submission
- ✅ Detailed error reporting

### Next Steps:
1. Try registering different types of API documentation
2. Experiment with various documentation formats
3. Check the API registry to see your registered APIs